# Time-Series Machine Learning — All Traditional Statistical Models (No `if`, No Baseline Functions)

This version differs from the first notebook in two ways, per request:

1. **No `if` statements anywhere** (including no "is the library installed" checks) — all libraries are installed and imported directly.
2. **Baseline models are computed inline** (Naive, Historical Average, Window Average, Seasonal Naive) instead of being wrapped in user-defined functions.

It also covers **every traditional statistical model** from the notes:
`ARIMA`, `SARIMA`, `Simple Exponential Smoothing`, `Holt`, `Holt-Winters`, `Theta`, `MSTL`,
`BATS`, `TBATS`, `Croston`, `IMAPA`, `TSB`.

> Run the install cell once (requires internet access in your environment).


## 1. Install & Import Libraries

In [ ]:
!pip install -q statsmodels tbats statsforecast


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.tsa.forecasting.theta import ThetaModel
from statsmodels.tsa.seasonal import seasonal_decompose, MSTL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from tbats import BATS, TBATS
from statsforecast.models import CrostonClassic, IMAPA, TSB

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (11, 4)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Create Synthetic Data

Two series:
- `sales` — a regular daily series with trend + weekly & yearly seasonality (for ARIMA/SARIMA/ETS/Theta/MSTL/BATS/TBATS/ML/baselines).
- `intermittent_demand` — a sparse series with mostly zeros (for Croston/IMAPA/TSB).


In [ ]:
n_days = 3 * 365
dates = pd.date_range("2022-01-01", periods=n_days, freq="D")
t = np.arange(n_days)

level = 100
trend = 0.05 * t
weekly_seasonality = 10 * np.sin(2 * np.pi * t / 7)
yearly_seasonality = 25 * np.sin(2 * np.pi * t / 365.25)
noise = np.random.normal(0, 5, n_days)

sales = np.maximum(level + trend + weekly_seasonality + yearly_seasonality + noise, 0)
price = 20 + np.random.normal(0, 1, n_days).cumsum() * 0.01
temperature = 15 + 10 * np.sin(2 * np.pi * (t + 80) / 365.25) + np.random.normal(0, 2, n_days)

df = pd.DataFrame({"date": dates, "sales": sales, "price": price, "temperature": temperature})

outlier_idx = np.random.choice(n_days, size=5, replace=False)
df.loc[outlier_idx, "sales"] *= 2.5

missing_idx = np.random.choice(n_days, size=8, replace=False)
df.loc[missing_idx, "sales"] = np.nan

df = df.set_index("date")

# Intermittent / sparse demand series (spare-parts style)
demand_prob = 0.12
demand_size = np.random.gamma(shape=3, scale=8, size=n_days)
occurrence = np.random.binomial(1, demand_prob, size=n_days)
intermittent_demand = pd.Series(occurrence * demand_size, index=dates, name="intermittent_demand")

df.head()


## 3. Exploratory Data Analysis

In [ ]:
df["sales"].plot(title="Daily Sales Over Time")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

df["sales"].describe()


In [ ]:
decomposition = seasonal_decompose(df["sales"].ffill(), model="additive", period=365)
fig = decomposition.plot()
fig.set_size_inches(11, 8)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df["sales"].ffill(), lags=40, ax=axes[0])
plot_pacf(df["sales"].ffill(), lags=40, ax=axes[1])
plt.tight_layout()
plt.show()


In [ ]:
intermittent_demand.plot(title="Intermittent Demand (sparse series)")
plt.ylabel("Demand")
plt.tight_layout()
plt.show()

print("Zero-demand periods:", (intermittent_demand == 0).mean())


## 4. Handle Missing Values

In [ ]:
df["sales_clean"] = df["sales"].interpolate(method="linear")
print("Remaining missing values:", df["sales_clean"].isna().sum())


## 5. Feature Engineering

In [ ]:
lags = [1, 2, 3, 7, 14, 28]
for lag in lags:
    df[f"lag_{lag}"] = df["sales_clean"].shift(lag)

windows = [7, 14, 30]
for w in windows:
    shifted = df["sales_clean"].shift(1)
    df[f"rolling_mean_{w}"] = shifted.rolling(window=w).mean()
    df[f"rolling_std_{w}"] = shifted.rolling(window=w).std()

df["day_of_week"] = df.index.dayofweek
df["day_of_month"] = df.index.day
df["month"] = df.index.month
df["quarter"] = df.index.quarter
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["day_of_week_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["day_of_week_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["diff_1"] = df["sales_clean"].diff(1)
df["diff_7"] = df["sales_clean"].diff(7)

df.tail()


## 6. Chronological Train / Validation / Test Split

In [ ]:
feature_cols = [
    "lag_1", "lag_2", "lag_3", "lag_7", "lag_14", "lag_28",
    "rolling_mean_7", "rolling_std_7", "rolling_mean_14", "rolling_mean_30",
    "day_of_week", "day_of_month", "month", "quarter", "is_weekend",
    "month_sin", "month_cos", "day_of_week_sin", "day_of_week_cos",
    "price", "temperature",
]
target_col = "sales_clean"

model_df = df.dropna(subset=feature_cols + [target_col]).copy()

train_end = "2023-12-31"
val_end = "2024-09-30"

train = model_df.loc[:train_end]
val = model_df.loc[pd.Timestamp(train_end) + pd.Timedelta(days=1):val_end]
test = model_df.loc[pd.Timestamp(val_end) + pd.Timedelta(days=1):]

full_history = pd.concat([train[target_col], val[target_col]])

print(f"Train: {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} rows)")
print(f"Val:   {val.index.min().date()} -> {val.index.max().date()}  ({len(val)} rows)")
print(f"Test:  {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} rows)")


## 7. Baseline Models (Inline — No Functions)

Each baseline is computed directly against the test horizon; nothing is wrapped in a `def`.


In [ ]:
test_len = len(test)
history_series = full_history

# Naive: repeat the last observed value
naive_pred = np.full(test_len, history_series.iloc[-1])

# Historical Average: mean of all past observations
historical_avg_pred = np.full(test_len, history_series.mean())

# Window Average: mean of the most recent 7 observations
window_avg_pred = np.full(test_len, history_series.iloc[-7:].mean())

# Seasonal Naive: repeat the last full week, tiled across the horizon
last_season = history_series.iloc[-7:].values
seasonal_naive_pred = np.tile(last_season, int(np.ceil(test_len / 7)))[:test_len]

y_test = test[target_col].values

baseline_rmse = pd.Series({
    "Naive": np.sqrt(mean_squared_error(y_test, naive_pred)),
    "Historical Average": np.sqrt(mean_squared_error(y_test, historical_avg_pred)),
    "Window Average (7)": np.sqrt(mean_squared_error(y_test, window_avg_pred)),
    "Seasonal Naive (7)": np.sqrt(mean_squared_error(y_test, seasonal_naive_pred)),
}, name="RMSE")

baseline_rmse


## 8. Traditional Statistical Models

### 8.1 ARIMA

In [ ]:
arima_fit = ARIMA(full_history, order=(2, 1, 2)).fit()
arima_forecast = arima_fit.forecast(steps=test_len)
arima_rmse = np.sqrt(mean_squared_error(y_test, arima_forecast))
arima_rmse


### 8.2 SARIMA

In [ ]:
sarima_fit = SARIMAX(
    full_history,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
sarima_forecast = sarima_fit.forecast(steps=test_len)
sarima_rmse = np.sqrt(mean_squared_error(y_test, sarima_forecast))
sarima_rmse


### 8.3 Simple Exponential Smoothing

In [ ]:
ses_fit = SimpleExpSmoothing(full_history).fit()
ses_forecast = ses_fit.forecast(test_len)
ses_rmse = np.sqrt(mean_squared_error(y_test, ses_forecast))
ses_rmse


### 8.4 Holt (level + trend)

In [ ]:
holt_fit = Holt(full_history).fit()
holt_forecast = holt_fit.forecast(test_len)
holt_rmse = np.sqrt(mean_squared_error(y_test, holt_forecast))
holt_rmse


### 8.5 Holt-Winters (level + trend + seasonality)

In [ ]:
hw_fit = ExponentialSmoothing(full_history, trend="add", seasonal="add", seasonal_periods=7).fit()
hw_forecast = hw_fit.forecast(test_len)
hw_rmse = np.sqrt(mean_squared_error(y_test, hw_forecast))
hw_rmse


### 8.6 Theta Model

In [ ]:
theta_fit = ThetaModel(full_history, period=7).fit()
theta_forecast = theta_fit.forecast(steps=test_len)
theta_rmse = np.sqrt(mean_squared_error(y_test, theta_forecast))
theta_rmse


### 8.7 MSTL (Multiple Seasonal-Trend decomposition using LOESS)

MSTL decomposes the series into trend + multiple seasonal components + remainder.
To forecast: extend the trend with Simple Exponential Smoothing, extend each seasonal
component by repeating its last full cycle, and sum the pieces together.


In [ ]:
mstl_fit = MSTL(full_history, periods=(7, 365)).fit()

trend_component = mstl_fit.trend
seasonal_7 = mstl_fit.seasonal["seasonal_7"]
seasonal_365 = mstl_fit.seasonal["seasonal_365"]

trend_forecast = SimpleExpSmoothing(trend_component).fit().forecast(test_len).values
seasonal_7_forecast = np.tile(seasonal_7.iloc[-7:].values, int(np.ceil(test_len / 7)))[:test_len]
seasonal_365_forecast = np.tile(seasonal_365.iloc[-365:].values, int(np.ceil(test_len / 365)))[:test_len]

mstl_forecast = trend_forecast + seasonal_7_forecast + seasonal_365_forecast
mstl_rmse = np.sqrt(mean_squared_error(y_test, mstl_forecast))
mstl_rmse


### 8.8 BATS

In [ ]:
bats_fit = BATS(seasonal_periods=[7]).fit(full_history.values)
bats_forecast = bats_fit.forecast(steps=test_len)
bats_rmse = np.sqrt(mean_squared_error(y_test, bats_forecast))
bats_rmse


### 8.9 TBATS

In [ ]:
tbats_fit = TBATS(seasonal_periods=[7, 365.25]).fit(full_history.values)
tbats_forecast = tbats_fit.forecast(steps=test_len)
tbats_rmse = np.sqrt(mean_squared_error(y_test, tbats_forecast))
tbats_rmse


### 8.10 Intermittent-Demand Models: Croston, IMAPA, TSB

These use the sparse `intermittent_demand` series instead of `sales`.


In [ ]:
intermittent_train = intermittent_demand.iloc[:-90]
intermittent_test = intermittent_demand.iloc[-90:]
intermittent_horizon = len(intermittent_test)

croston_model = CrostonClassic()
croston_model = croston_model.fit(y=intermittent_train.values)
croston_forecast = croston_model.predict(h=intermittent_horizon)["mean"]
croston_rmse = np.sqrt(mean_squared_error(intermittent_test.values, croston_forecast))

imapa_model = IMAPA()
imapa_model = imapa_model.fit(y=intermittent_train.values)
imapa_forecast = imapa_model.predict(h=intermittent_horizon)["mean"]
imapa_rmse = np.sqrt(mean_squared_error(intermittent_test.values, imapa_forecast))

tsb_model = TSB(alpha_d=0.1, alpha_p=0.1)
tsb_model = tsb_model.fit(y=intermittent_train.values)
tsb_forecast = tsb_model.predict(h=intermittent_horizon)["mean"]
tsb_rmse = np.sqrt(mean_squared_error(intermittent_test.values, tsb_forecast))

intermittent_rmse = pd.Series({
    "Croston": croston_rmse,
    "IMAPA": imapa_rmse,
    "TSB": tsb_rmse,
}, name="RMSE (intermittent demand)")

intermittent_rmse


## 9. Walk-Forward Validation (Expanding & Rolling — No Branching)

Two separate loops, one per window type, instead of a single function with `if/elif`.


In [ ]:
initial_train_size = 730
series = model_df[target_col]

expanding_dates, expanding_actual, expanding_pred = [], [], []
for cutoff in range(initial_train_size, len(series)):
    train_window = series.iloc[:cutoff]
    expanding_pred.append(train_window.iloc[-1])
    expanding_actual.append(series.iloc[cutoff])
    expanding_dates.append(series.index[cutoff])

expanding_results = pd.DataFrame(
    {"actual": expanding_actual, "predicted": expanding_pred}, index=expanding_dates
)
expanding_rmse = np.sqrt(mean_squared_error(expanding_results["actual"], expanding_results["predicted"]))
expanding_rmse


In [ ]:
rolling_window_size = 90

rolling_dates, rolling_actual, rolling_pred = [], [], []
for cutoff in range(initial_train_size, len(series)):
    train_window = series.iloc[cutoff - rolling_window_size:cutoff]
    rolling_pred.append(train_window.iloc[-1])
    rolling_actual.append(series.iloc[cutoff])
    rolling_dates.append(series.index[cutoff])

rolling_results = pd.DataFrame(
    {"actual": rolling_actual, "predicted": rolling_pred}, index=rolling_dates
)
rolling_rmse = np.sqrt(mean_squared_error(rolling_results["actual"], rolling_results["predicted"]))
rolling_rmse


## 10. Machine-Learning Model (Gradient Boosting)

In [ ]:
X_train, y_train_arr = train[feature_cols], train[target_col]
X_test = test[feature_cols]

gbm = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_SEED)
gbm.fit(X_train, y_train_arr)
gbm_forecast = gbm.predict(X_test)
gbm_rmse = np.sqrt(mean_squared_error(y_test, gbm_forecast))
gbm_rmse


In [ ]:
importances = pd.Series(gbm.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.plot(kind="barh", figsize=(8, 6), title="Gradient Boosting Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 11. Model Comparison

In [ ]:
comparison_df = pd.DataFrame({
    "RMSE": {
        "Naive": baseline_rmse["Naive"],
        "Historical Average": baseline_rmse["Historical Average"],
        "Window Average (7)": baseline_rmse["Window Average (7)"],
        "Seasonal Naive (7)": baseline_rmse["Seasonal Naive (7)"],
        "ARIMA": arima_rmse,
        "SARIMA": sarima_rmse,
        "Simple Exp. Smoothing": ses_rmse,
        "Holt": holt_rmse,
        "Holt-Winters": hw_rmse,
        "Theta": theta_rmse,
        "MSTL": mstl_rmse,
        "BATS": bats_rmse,
        "TBATS": tbats_rmse,
        "Gradient Boosting": gbm_rmse,
    }
}).sort_values("RMSE")

comparison_df


In [ ]:
comparison_df.plot(kind="bar", legend=False, title="Model Comparison — Test RMSE (lower is better)")
plt.ylabel("RMSE")
plt.tight_layout()
plt.show()

print("Intermittent-demand models (separate series, not comparable to RMSE above):")
intermittent_rmse


## 12. Key Takeaways

- **Baselines** (Naive, Historical Average, Window Average, Seasonal Naive) are computed directly, inline — a complex model that can't beat them isn't adding value.
- **Every traditional statistical model** from the notes is included: ARIMA, SARIMA, Simple Exponential Smoothing, Holt, Holt-Winters, Theta, MSTL, BATS, TBATS, plus the intermittent-demand specialists Croston, IMAPA, and TSB.
- **Walk-forward validation** is implemented as two independent loops (expanding vs. rolling) rather than one branching function.
- Split chronologically, engineer temporal features without leakage, and monitor for concept drift after deployment.
